In [ ]:
# Path configuration
out <- 'outputs/entry'


# Entry regressions with relatedness density

Run from `revision_notebooks/` using an R kernel after notebook 09. Install `arrow` and **fixest 0.12.1**. Fit the source-field and GBB model on the full entry sample, then compare three models on the common sample for which relatedness density can be calculated.

All models include firm and CCMT-class fixed effects, with standard errors clustered by firm and CCMT class. In fixest 0.12.1, `fixef.K="nested"` counts nonnested fixed effects for the small-sample correction.


In [ ]:
library(arrow)
library(fixest)
dir.create(out, recursive=TRUE, showWarnings=FALSE)

## Full entry sample and common sample

The full sample contains the firm–CCMT observations selected in notebook 09. The common sample contains observations with a measured relatedness density. Use this same sample for the density-only model, the source-field and GBB model, and their combination.


In [ ]:
d <- as.data.frame(read_parquet(file.path(out, 'entry_density.parquet')))
common <- d[d$portfolio_observed & is.finite(d$proximity_density), ]

## Fit the four specifications


In [ ]:
# fixest 0.12.1 names the nonnested-FE counting rule fixef.K="nested".
ssc_rule <- ssc(adj=TRUE, fixef.K='nested', fixef.force_exact=FALSE,
                cluster.adj=TRUE, cluster.df='min', t.df='min')
original <- I(num_pat_ccmt_2 > 0) ~ log1p(path) * log1p(src) | person_id + y02
models <- list(
  original_full=feols(original, d, cluster=~person_id+y02, ssc=ssc_rule, lean=TRUE),
  density_only=feols(I(num_pat_ccmt_2 > 0) ~ proximity_density | person_id + y02,
                    common, cluster=~person_id+y02, ssc=ssc_rule, lean=TRUE),
  original_common=feols(original, common, cluster=~person_id+y02, ssc=ssc_rule, lean=TRUE),
  density=feols(I(num_pat_ccmt_2 > 0) ~ log1p(path) * log1p(src) + proximity_density |
                 person_id + y02, common, cluster=~person_id+y02, ssc=ssc_rule, lean=TRUE))

## Save estimates, confidence intervals, sample counts and tables


In [ ]:
coefficients <- lapply(names(models), function(id) {
  m <- models[[id]]; ct <- coeftable(m); ci <- confint(m)
  data.frame(model=id, term=rownames(ct), estimate=ct[,1], std_error=ct[,2],
             statistic=ct[,3], p_value=ct[,4], conf_low=ci[,1], conf_high=ci[,2], row.names=NULL)
})
write.csv(do.call(rbind, coefficients), file.path(out, 'coefficients.csv'), row.names=FALSE)
metrics <- lapply(names(models), function(id) data.frame(model=id, nobs=nobs(models[[id]]),
  within_r2=as.numeric(fitstat(models[[id]], 'wr2')[[1]]),
  cluster_t_df=degrees_freedom(models[[id]], 't')))
write.csv(do.call(rbind, metrics), file.path(out, 'model_metrics.csv'), row.names=FALSE)
coverage <- function(sample, x) data.frame(sample=sample, rows=nrow(x),
  firms=length(unique(x$person_id)), targets=length(unique(x$y02)), events=sum(x$num_pat_ccmt_2 > 0))
write.csv(rbind(coverage('original_full',d), coverage('density_supported',common)),
          file.path(out,'sample_coverage.csv'), row.names=FALSE)
writeLines(as.character(etable(models['original_full'], tex=TRUE)), file.path(out,'original_full.tex'))
writeLines(as.character(etable(models[c('density_only','original_common','density')], tex=TRUE)),
           file.path(out,'entry_density.tex'))
writeLines(capture.output(sessionInfo()), file.path(out, 'session_info.txt'))

etable(models)